In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/nirmalsankalana
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Tomato verticulium wilt
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cassava green mite
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cassava mosaic
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cashew red rust
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cashew gumosis
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Tomato healthy
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cassava brown spot
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Cassava bacterial blight
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle
/kaggle/input/datasets/nirmalsankalana/crop-pest

In [4]:
import torch
import torchvision

print(torch.__version__)

2.10.0+cu128


In [5]:
dataset_path = "/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection"

In [6]:
import os

dataset_path = "/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection"

print("Number of classes:", len(os.listdir(dataset_path)))
print(os.listdir(dataset_path))

Number of classes: 22
['Tomato verticulium wilt', 'Cassava green mite', 'Cassava mosaic', 'Cashew red rust', 'Cashew gumosis', 'Tomato healthy', 'Cassava brown spot', 'Cassava bacterial blight', 'Maize leaf beetle', 'Cassava healthy', 'Maize leaf spot', 'Maize healthy', 'Tomato leaf blight', 'Cashew healthy', 'Cashew leaf miner', 'Maize streak virus', 'Tomato septoria leaf spot', 'Maize leaf blight', 'Maize grasshoper', 'Cashew anthracnose', 'Tomato leaf curl', 'Maize fall armyworm']


In [7]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

print("Total Images :", len(dataset))
print("Number of Classes :", len(dataset.classes))
print("Classes :")
print(dataset.classes)

Total Images : 25220
Number of Classes : 22
Classes :
['Cashew anthracnose', 'Cashew gumosis', 'Cashew healthy', 'Cashew leaf miner', 'Cashew red rust', 'Cassava bacterial blight', 'Cassava brown spot', 'Cassava green mite', 'Cassava healthy', 'Cassava mosaic', 'Maize fall armyworm', 'Maize grasshoper', 'Maize healthy', 'Maize leaf beetle', 'Maize leaf blight', 'Maize leaf spot', 'Maize streak virus', 'Tomato healthy', 'Tomato leaf blight', 'Tomato leaf curl', 'Tomato septoria leaf spot', 'Tomato verticulium wilt']


In [8]:
from torch.utils.data import random_split, DataLoader

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(f"Training Images: {len(train_dataset)}")
print(f"Validation Images: {len(val_dataset)}")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders Created Successfully!")

Training Images: 20176
Validation Images: 5044
DataLoaders Created Successfully!


In [9]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load pretrained model
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace classifier
num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

model = model.to(device)

print(model)

Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 148MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [10]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [11]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss and Optimizer Ready!")

Loss and Optimizer Ready!


In [12]:
from PIL import Image
import os

dataset_path = "/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection"

bad_images = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)
        try:
            img = Image.open(path)
            img.verify()   # Verify image integrity
        except Exception:
            bad_images.append(path)

print("Corrupted Images:", len(bad_images))

for img in bad_images:
    print(img)

Corrupted Images: 50
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Tomato healthy/healthy443_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Tomato healthy/healthy77_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle90_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle208_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle458_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle690_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle326_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle573_.jpg
/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection/Maize leaf beetle/leaf beetle207_.jpg
/kaggle/input/

In [13]:
import shutil

src = "/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection"
dst = "/kaggle/working/crop-pest-and-disease-detection"

shutil.copytree(src, dst)

print("Dataset copied successfully!")

Dataset copied successfully!


In [14]:
import os

for img in bad_images:
    new_path = img.replace(
        "/kaggle/input/datasets/nirmalsankalana/crop-pest-and-disease-detection",
        "/kaggle/working/crop-pest-and-disease-detection"
    )

    if os.path.exists(new_path):
        os.remove(new_path)

print("All corrupted images removed!")

All corrupted images removed!


In [15]:
dataset_path = "/kaggle/working/crop-pest-and-disease-detection"

In [16]:
from torchvision import datasets, transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=train_transform
)

print("Total Images:", len(dataset))
print("Classes:", len(dataset.classes))

Total Images: 25170
Classes: 22


In [17]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 20136
Validation: 5034


In [18]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    len(dataset.classes)
)

model = model.to(device)

print("Device:", device)

Device: cuda


In [19]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss Function Ready")
print("Optimizer Ready")

Loss Function Ready
Optimizer Ready


In [20]:
from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

In [21]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Loss     : {epoch_loss:.4f}")
    print(f"Accuracy : {epoch_acc:.2f}%")
    print("-" * 40)

Epoch [1/10]
Loss     : 0.9093
Accuracy : 73.62%
----------------------------------------
Epoch [2/10]
Loss     : 0.4004
Accuracy : 85.97%
----------------------------------------
Epoch [3/10]
Loss     : 0.3066
Accuracy : 89.09%
----------------------------------------
Epoch [4/10]
Loss     : 0.2528
Accuracy : 90.93%
----------------------------------------
Epoch [5/10]
Loss     : 0.2185
Accuracy : 92.11%
----------------------------------------
Epoch [6/10]
Loss     : 0.1893
Accuracy : 93.32%
----------------------------------------
Epoch [7/10]
Loss     : 0.1617
Accuracy : 93.96%
----------------------------------------
Epoch [8/10]
Loss     : 0.1431
Accuracy : 94.63%
----------------------------------------
Epoch [9/10]
Loss     : 0.1283
Accuracy : 95.39%
----------------------------------------
Epoch [10/10]
Loss     : 0.1143
Accuracy : 95.91%
----------------------------------------


In [22]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

val_acc = 100 * correct / total

print(f"Validation Accuracy: {val_acc:.2f}%")

Validation Accuracy: 90.82%


In [23]:
import torch

torch.save(model.state_dict(), "/kaggle/working/leaf_model.pth")

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [24]:
import json

class_names = dataset.classes

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)

print(class_names)
print("✅ Class names saved!")

['Cashew anthracnose', 'Cashew gumosis', 'Cashew healthy', 'Cashew leaf miner', 'Cashew red rust', 'Cassava bacterial blight', 'Cassava brown spot', 'Cassava green mite', 'Cassava healthy', 'Cassava mosaic', 'Maize fall armyworm', 'Maize grasshoper', 'Maize healthy', 'Maize leaf beetle', 'Maize leaf blight', 'Maize leaf spot', 'Maize streak virus', 'Tomato healthy', 'Tomato leaf blight', 'Tomato leaf curl', 'Tomato septoria leaf spot', 'Tomato verticulium wilt']
✅ Class names saved!
